##Ingestion + Bronze

In [0]:

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RAW = "/Volumes/jrvs_databricks_fundamentals/default/raw"

def create_bronze_table(table_name, subfolder):
    @dp.table(name=table_name)
    def ingest():
        return (
            spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("cloudFiles.inferColumnTypes", "false")
            .load(f"{RAW}/{subfolder}")
            .withColumn("_ingest_ts", F.col("_metadata.file_modification_time")) 
            #mental note: autoloaders hidden columns, file_modification_time is the ingestion timestamp
            # .withColumn("_source_file", F.col("_metadata.file_path")) 
        )
    return ingest

create_bronze_table("daily_bronze", "daily")
create_bronze_table("quote_bronze", "quote")
create_bronze_table("company_bronze", "company")

##Silver

In [0]:

#Helper Function
def clean_str(column_name):
    nulls = ["None", "", "-", "N/A"]
    value = F.trim(F.col(column_name))
    return F.when(value.isin(nulls), None).otherwise(value)

@dp.view(name="daily_silver_clean")
def daily_silver_clean():
    return (
        spark.readStream.table("daily_bronze")
            .withColumn("trade_date", F.to_date("trade_date"))
            .withColumn("open", F.round(F.col("open").cast("double"), 2))
            .withColumn("high", F.round(F.col("high").cast("double"), 2))
            .withColumn("low", F.round(F.col("low").cast("double"), 2))
            .withColumn("close", F.round(F.col("close").cast("double"), 2))
            .withColumn("volume", F.col("volume").cast("long"))
    )

dp.create_streaming_table(
    name="daily_silver",
    expect_all_or_drop={
        "has_symbol":     "symbol IS NOT NULL",
        "has_date":       "trade_date IS NOT NULL",
        "close_positive": "close > 0",
    },
)

dp.apply_changes(
    target      = "daily_silver",
    source      = "daily_silver_clean",
    keys        = ["symbol", "trade_date"],
    sequence_by = F.col("_ingest_ts"),
    ignore_null_updates = True,
    stored_as_scd_type  = 1,
)


@dp.view(name="quote_silver_clean")
def quote_silver_clean():
    return (
        spark.readStream.table("quote_bronze")
            .withColumn("latest_trading_day", F.to_date("latest_trading_day"))
            .withColumn("price", F.round(F.col("price").cast("double"), 2))
            .withColumn("previous_close", F.round(F.col("previous_close").cast("double"), 2))
            .withColumn("price_change", F.round(F.col("price_change").cast("double"), 2))
            # change percent = (price − previous_close) / previous_close × 100
            .withColumn("change_percent",
                F.round((F.col("price") - F.col("previous_close")) / F.col("previous_close") * 100, 2))
            .withColumn("volume", F.col("volume").cast("double"))
    )

dp.create_streaming_table(
    name="quote_silver",
    expect_all_or_drop={
        "has_symbol": "symbol IS NOT NULL",
        "has_date":   "latest_trading_day IS NOT NULL",
    },
)

dp.apply_changes(
    target      = "quote_silver",
    source      = "quote_silver_clean",
    keys        = ["symbol"],
    sequence_by = F.col("_ingest_ts"),
    ignore_null_updates = True,
    stored_as_scd_type  = 1,
)

@dp.view(name="company_silver_clean")
def company_silver_clean():
    return (
        spark.readStream.table("company_bronze")
            .withColumn("symbol", clean_str("symbol"))
            .withColumn("company_name", clean_str("company_name"))
            .withColumn("sector", clean_str("sector"))
            .withColumn("industry", clean_str("industry"))
            .withColumn("country", clean_str("country"))
            .withColumn("exchange", clean_str("exchange"))
            .withColumn("market_cap", clean_str("market_cap").cast("double"))
    )

dp.create_streaming_table(
    name="company_silver",
    expect_all_or_drop={"has_symbol": "symbol IS NOT NULL"},
)

dp.apply_changes(
    target      = "company_silver",
    source      = "company_silver_clean",
    keys        = ["symbol"],
    sequence_by = F.col("_ingest_ts"),
    stored_as_scd_type = 2,
    track_history_except_column_list = ["market_cap"],
    except_column_list = ["_ingest_ts"],
)

##Gold

In [0]:

@dp.table(name="daily_gold")
def daily_gold():
    s = spark.read.table("daily_silver")
    w = Window.partitionBy("symbol").orderBy("trade_date")
    out = s
    for n in (7, 30, 90):
        prev_close  = F.lag("close", n).over(w)
        prev_volume = F.lag("volume", n).over(w)
        out = (
            out
            .withColumn(f"price_change_{n}d",     F.round(F.col("close") - prev_close, 2))
            .withColumn(f"price_pct_change_{n}d", F.round((F.col("close") - prev_close) / prev_close * 100, 2))
            .withColumn(f"volume_change_{n}d",    F.col("volume") - prev_volume)
        )
    return out


@dp.table(name="quote_gold")
def quote_gold():
    return (
        spark.read.table("quote_silver")
             .select("symbol", "price", "previous_close", "price_change",
                     "change_percent", "volume", "latest_trading_day")
    )


@dp.table(name="company_gold")
def company_gold():
    return (
        #SCD2, curr = __END = Null for current 
        spark.read.table("company_silver")
             .filter(F.col("__END_AT").isNull())  
             .select("symbol", "company_name", "sector", "industry",
                     "country", "exchange", "market_cap")
    )